# DistilRoBERTa Fine-Tuning Baseline (Naeem): Task #2

<a href="https://colab.research.google.com/github/Break-Through-Tech/Accenture-1F-contract-review-challenge/blob/main/notebooks/naeem_distilroberta_finetune_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Milestone #2 (October), **Task #2**: finalize chunking, prepare the tokenized dataset, set up the training pipeline, and fine-tune a baseline.

- **Chunking:** Week 3 paragraph-aware `paragraph_tok_480_overlap_64`, re-run with the `distilroberta-base` tokenizer
- **Labels:** multi-label, one per CUAD category. Full containment counts as positive; partial-only overlap is masked
- **Model:** `distilroberta-base` + sigmoid heads, masked BCE loss
- **Split:** official CUAD train/test (408 / 102 contracts); 10% of train contracts held out for validation

Code lives in `training/` (see `training/README.md`). **Before running:** Runtime → Change runtime type → **T4 GPU**.

In [ ]:
# 1. Get the repo (re-running pulls the latest main)
import os
REPO = "Accenture-1F-contract-review-challenge"
if not os.path.exists(REPO):
    !git clone -q https://github.com/Break-Through-Tech/{REPO}.git
else:
    !git -C {REPO} pull -q
%cd {REPO}
!pip install -q -r training/requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Finalize chunking + build the tokenized dataset
Takes a few minutes on CPU. Writes to `training/data/`.

In [ ]:
!python training/prepare_dataset.py

In [ ]:
import pandas as pd
summary = pd.read_csv("training/data/dataset_summary.csv")
summary.pivot(index="category", columns="split", values="positive_chunks").sort_values("train", ascending=False)

## 3. Fine-tune DistilRoBERTa
Roughly 3–5 min per epoch on a T4. For a quick check, add `--epochs 1 --limit 256`.
Add `--use-pos-weight` to try per-category positive weighting (Task #3).

In [ ]:
!python training/train_distilroberta.py --epochs 3 --batch-size 16 --lr 3e-5

## 4. Results

In [ ]:
history = pd.read_csv("training/outputs/history.csv")
chunk_metrics = pd.read_csv("training/outputs/chunk_level_metrics.csv")
contract_metrics = pd.read_csv("training/outputs/contract_level_metrics.csv")
display(history)
display(chunk_metrics.sort_values("F1_Score", ascending=False))

In [ ]:
# Contract-level view on the same 7 categories as the TF-IDF baseline notebook
tfidf_categories = [
    "Anti-Assignment", "Change Of Control", "Insurance", "Renewal Term",
    "Notice Period To Terminate Renewal", "Audit Rights", "Termination For Convenience",
]
display(contract_metrics[contract_metrics["Category"].isin(tfidf_categories)])

## 5. (Optional) Save the model to Google Drive
Colab storage is wiped when the runtime disconnects.

In [ ]:
# from google.colab import drive
# drive.mount("/content/drive")
# !mkdir -p /content/drive/MyDrive/accenture-1f && cp -r training/outputs /content/drive/MyDrive/accenture-1f/distilroberta_baseline